In [0]:
pip install tqdm

In [0]:
pip install torch

In [0]:
from pyspark.sql.functions import *
import os
from functools import reduce
from pyspark.sql import DataFrame
from pyspark.sql.window import Window
from pyspark.sql.types import TimestampType, StructType, StructField, ArrayType, DoubleType, IntegerType
import pandas as pd
from pyarrow import *
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression  
from pyspark.ml.feature import *
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np
import pyarrow.parquet as pq
from pyspark.storagelevel import *
from math import *
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer
from pyspark.ml.pipeline import PipelineModel

In [0]:
lstm = spark.read.table("hive_metastore.default.lreg_df")

In [0]:
lstm = lstm.withColumn("AMPM_flag", when(col("AM_PM")=="PM", 1.0).otherwise(0.0))

In [0]:
display(lstm
    .groupBy("has_falha")
    .agg(F.count("*").alias("count"))
    .orderBy("has_falha"))

In [0]:
# =========================
# 0) CONFIG
# =========================
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer
from pyspark.ml.pipeline import PipelineModel
from pyspark.sql.window import Window

# Base numeric columns (no weather)
NUMERIC_COLS = [
     "INTENSITY","TENSION","H_LIM_I","H_LIM_T",
     "MAVERAGE_2H_I","MAVERAGE_2H_T","MAVERAGE_1D_I","MAVERAGE_1D_T",
     "EVENT_COUNT_I","EVENT_COUNT_T",
     "TIME_OVER_LIMIT_I","TIME_OVER_LIMIT_T",
     "DAY_OF_MONTH","DAY_OF_YEAR","HOUR_OF_DAY","DAY_OF_WEEK",
     "AMPM_flag"
]
CAT_COLS = ["ID","CONCELHO"]

CUT_OFF    = "2023-11-30"
SAVE_PATH  = "dbfs:/models/lstm_indexers_v1"
NSTEPS     = 96
VAL_DAYS   = 2                    # final small VAL window before CUT_OFF
K_FOLDS    = 3                    # walk-forward folds for HPO
CADENCE_MIN= 15                   # 15-min cadence
PURGE_STEPS= NSTEPS - 1           # leakage purge for folds

ID_COL    = "ID"
LABEL_COL = "has_falha"


In [0]:
# =========================
# 1) BASE SELECT + CYCLICAL
# =========================
SELECT_COLS = [ID_COL,"DATE",LABEL_COL] + [c for c in CAT_COLS if c != ID_COL] + NUMERIC_COLS

base0 = (lstm
    .select(*[F.col(c) for c in SELECT_COLS])
    .withColumn("DATE", F.col("DATE").cast("timestamp"))
    .withColumnRenamed(LABEL_COL, "label")
)
display(base0.limit(10))

pi = 3.141592653589793
df_time = (base0
    .withColumn("DOW_SIN",  F.sin(2*pi * (F.col("DAY_OF_WEEK")/7.0)))
    .withColumn("DOW_COS",  F.cos(2*pi * (F.col("DAY_OF_WEEK")/7.0)))
    .withColumn("HOUR_SIN", F.sin(2*pi * (F.col("HOUR_OF_DAY")/24.0)))
    .withColumn("HOUR_COS", F.cos(2*pi * (F.col("HOUR_OF_DAY")/24.0)))
    .withColumn("DOM_SIN",  F.sin(2*pi * (F.col("DAY_OF_MONTH")/31.0)))
    .withColumn("DOM_COS",  F.cos(2*pi * (F.col("DAY_OF_MONTH")/31.0)))
    .withColumn("DOY_SIN",  F.sin(2*pi * (F.col("DAY_OF_YEAR")/365.0)))
    .withColumn("DOY_COS",  F.cos(2*pi * (F.col("DAY_OF_YEAR")/365.0)))
    .drop("DAY_OF_WEEK","HOUR_OF_DAY","DAY_OF_MONTH","DAY_OF_YEAR")
)
display(df_time.limit(20))


In [0]:
# =========================
# 2) STRING INDEXERS (fit on TRAIN only)
# =========================
train_df = df_time.filter(F.col("DATE") < F.lit(CUT_OFF))

def build_indexer_pipeline(cat_cols):
    return Pipeline(stages=[
        StringIndexer(inputCol=c, outputCol=f"{c}_INDEX",
                      handleInvalid="keep", stringOrderType="frequencyDesc")
        for c in cat_cols
    ])

print("No saved indexers found. Fitting on TRAIN split…")
indexer_model = build_indexer_pipeline(CAT_COLS).fit(train_df)
indexer_model.write().overwrite().save(SAVE_PATH)
print(f"Saved indexer pipeline to {SAVE_PATH}")

df_idx = indexer_model.transform(df_time)

for st in indexer_model.stages:
    cname = st.getInputCol()
    labs = list(st.labels)[:12]
    print(f"{cname} → {cname}_INDEX | sample:", list(enumerate(labs)))
display(df_idx.select("ID","ID_INDEX","CONCELHO","CONCELHO_INDEX").limit(20))


In [0]:
# =========================
# 3) SCALING (train-only z-score; names unchanged)
# =========================
# Numeric final = original numerics minus raw time ints + cyclical pairs
NUMERIC_FINAL = [
    "INTENSITY","TENSION","H_LIM_I","H_LIM_T",
    "MAVERAGE_2H_I","MAVERAGE_2H_T","MAVERAGE_1D_I","MAVERAGE_1D_T",
    "EVENT_COUNT_I","EVENT_COUNT_T","AMPM_flag",
    "TIME_OVER_LIMIT_I","TIME_OVER_LIMIT_T",
    "HOUR_SIN","HOUR_COS","DOW_SIN","DOW_COS","DOM_SIN","DOM_COS","DOY_SIN","DOY_COS"
]
CAT_IDX_COLS = [f"{c}_INDEX" for c in CAT_COLS]
FEATURES_FOR_LSTM = NUMERIC_FINAL + CAT_IDX_COLS
N_NUM = len(NUMERIC_FINAL); N_CAT = len(CAT_IDX_COLS); NFEAT = len(FEATURES_FOR_LSTM)
print("Numeric dims:", N_NUM, "| Categorical count:", N_CAT, "| Total INPUT_SIZE:", NFEAT)

# scale all numerics except binary/cyclical; event counts via log1p+z
DO_NOT_SCALE = {"AMPM_flag","HOUR_SIN","HOUR_COS","DOW_SIN","DOW_COS","DOM_SIN","DOM_COS","DOY_SIN","DOY_COS"}
NUMERIC_TO_SCALE = [c for c in NUMERIC_FINAL if c not in DO_NOT_SCALE]
LOG1P_THEN_SCALE = [c for c in ["EVENT_COUNT_I","EVENT_COUNT_T"] if c in NUMERIC_TO_SCALE]
NUMERIC_TO_SCALE = [c for c in NUMERIC_TO_SCALE if c not in LOG1P_THEN_SCALE]

train_df_for_stats = df_idx.filter(F.col("DATE") < F.lit(CUT_OFF))
agg_exprs  = [F.avg(c).alias(f"{c}__mu") for c in NUMERIC_TO_SCALE]
agg_exprs += [F.stddev_pop(c).alias(f"{c}__sd") for c in NUMERIC_TO_SCALE]
agg_exprs += [F.avg(F.log1p(F.col(c))).alias(f"{c}__mu_log1p") for c in LOG1P_THEN_SCALE]
agg_exprs += [F.stddev_pop(F.log1p(F.col(c))).alias(f"{c}__sd_log1p") for c in LOG1P_THEN_SCALE]
stats = train_df_for_stats.select(*agg_exprs).first().asDict()

def _safe(v, default=1.0): return float(v) if v not in (None, 0.0) else float(default)

for c in NUMERIC_TO_SCALE:
    mu = float(stats.get(f"{c}__mu", 0.0)); sd = _safe(stats.get(f"{c}__sd"), 1.0)
    df_idx = df_idx.withColumn(c, (F.col(c) - F.lit(mu)) / F.lit(sd))
for c in LOG1P_THEN_SCALE:
    mu = float(stats.get(f"{c}__mu_log1p", 0.0)); sd = _safe(stats.get(f"{c}__sd_log1p"), 1.0)
    df_idx = df_idx.withColumn(c, (F.log1p(F.col(c)) - F.lit(mu)) / F.lit(sd))


In [0]:
# =========================
# 4) CARDINALITIES + META
# =========================
maxes = (df_idx.filter(F.col("DATE") < F.lit(CUT_OFF))
              .agg(*[F.max(c).alias(c) for c in CAT_IDX_COLS])
              .collect()[0])

cat_cardinalities = {f"{c}_INDEX" if not c.endswith("_INDEX") else c:
                     (int(maxes[f"{c}_INDEX"]) + 1 if maxes[f"{c}_INDEX"] is not None else 1)
                     for c in CAT_COLS}

print("Cat cardinalities:", cat_cardinalities)

import os, json
META_DIR  = "dbfs:/models/lstm_indexers_v1"
META_DIR_LOCAL = META_DIR.replace("dbfs:/","/dbfs/")
dbutils.fs.mkdirs(META_DIR)
os.makedirs(META_DIR_LOCAL, exist_ok=True)
dbfs_meta = f"{META_DIR_LOCAL}/meta.json"

with open(dbfs_meta, "w") as f:
    json.dump({
        "FEATURES_FOR_LSTM": FEATURES_FOR_LSTM,
        "N_NUM": N_NUM,
        "N_CAT": N_CAT,
        "CAT_COLS": CAT_COLS,
        "CAT_IDX_COLS": CAT_IDX_COLS,
        "cat_cardinalities": cat_cardinalities
    }, f, indent=2)
print("Saved meta to", dbfs_meta)


In [0]:
# =========================
# 5) SEQUENCES (keep your window approach)
# =========================
feat_expr = F.array(*[F.col(c).cast("double") for c in FEATURES_FOR_LSTM])
dfF = (df_idx
       .select("ID","DATE","label", feat_expr.alias("f"))
       .withColumn("ts", F.col("DATE").cast("long")))

w = Window.partitionBy("ID").orderBy("ts").rowsBetween(-NSTEPS+1, 0)
df_seq = (dfF
          .withColumn("features", F.collect_list("f").over(w))
          .where(F.size("features") == NSTEPS)
          .select("ID","DATE","label","features"))
display(df_seq.limit(10))


In [0]:
# =========================
# 6) FINAL TRAIN / VAL / TEST SPLIT + WRITE
# =========================
val_start = spark.sql(f"select date_sub(to_date('{CUT_OFF}'), {VAL_DAYS}) as d").first()['d']

df_train_final = df_seq.filter(F.col("DATE") < F.lit(val_start)).select("features","label")
df_val_final   = df_seq.filter((F.col("DATE") >= F.lit(val_start)) & (F.col("DATE") < F.lit(CUT_OFF))).select("features","label")
df_test        = df_seq.filter(F.col("DATE") >= F.lit(CUT_OFF)).select("features","label")

display(df_train_final.select(F.size("features").alias("T"), F.size(F.col("features")[0]).alias("F")).summary())
display(df_val_final.select(F.size("features").alias("T"), F.size(F.col("features")[0]).alias("F")).summary())
display(df_test.select(F.size("features").alias("T"), F.size(F.col("features")[0]).alias("F")).summary())

TRAIN_PATH = f"dbfs:/lstm_train_96x{NFEAT}"
VAL_PATH   = f"dbfs:/lstm_val_96x{NFEAT}"
TEST_PATH  = f"dbfs:/lstm_test_96x{NFEAT}"

spark.conf.set("spark.sql.files.maxRecordsPerFile", 300000)
df_train_final.repartition(200).write.mode("overwrite").option("compression","snappy").parquet(TRAIN_PATH)
df_val_final.repartition(60).write.mode("overwrite").option("compression","snappy").parquet(VAL_PATH)
df_test.repartition(60).write.mode("overwrite").option("compression","snappy").parquet(TEST_PATH)
print("Wrote:", TRAIN_PATH, VAL_PATH, TEST_PATH)


In [0]:
# =========================
# 7) WALK-FORWARD FOLDS (TimeSeriesSplit) — minimal add-on
# =========================
from datetime import datetime, timezone

gap_sec = (PURGE_STEPS * CADENCE_MIN * 60)

fold_paths = []
for i in range(K_FOLDS):
    vs = spark.sql(f"select date_sub(to_date('{CUT_OFF}'), {(K_FOLDS-i)*VAL_DAYS}) d").first()['d']   # date
    ve = spark.sql(f"select date_add(to_date('{vs}'), {VAL_DAYS}) d").first()['d']                   # date

    # convert date -> UTC midnight epoch seconds
    vs_ts = int(datetime(vs.year, vs.month, vs.day, tzinfo=timezone.utc).timestamp())

    df2 = df_seq.withColumn("ts", F.col("DATE").cast("long"))
    tr  = (df2.filter(F.col("DATE") < F.lit(vs))                                   # by date
               .filter(F.col("ts") <= (F.lit(vs_ts) - F.lit(gap_sec)))              # purge gap
               .select("features","label"))
    va  = (df2.filter((F.col("DATE") >= F.lit(vs)) & (F.col("DATE") < F.lit(ve)))  # val window
               .select("features","label"))

    TRP = f"dbfs:/lstm_train_fold{i+1}_96x{NFEAT}"
    VAP = f"dbfs:/lstm_val_fold{i+1}_96x{NFEAT}"
    tr.repartition(120).write.mode("overwrite").parquet(TRP)
    va.repartition(40).write.mode("overwrite").parquet(VAP)
    fold_paths.append((TRP, VAP))


display(spark.createDataFrame([(i+1, trp, vap) for i,(trp,vap) in enumerate(fold_paths)],
                              ["fold","train_path","val_path"]))


In [0]:
# =========================
# 8) TRAINING (minimal tweaks; auto-use VAL if present)
# =========================
import os, json, math, numpy as np, sys, gc, time, builtins
import torch, torch.nn as nn, torch.optim as optim
import pyarrow.dataset as ds
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, accuracy_score

os.environ["TQDM_NOTEBOOK"] = "0"
from tqdm import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
META_PATH = "/dbfs/models/lstm_indexers_v1/meta.json"

def to_local_dbfs(p: str) -> str: return p.replace("dbfs:/", "/dbfs/")

TRAIN_PATH_LOCAL = to_local_dbfs(TRAIN_PATH)
VAL_PATH_LOCAL   = to_local_dbfs(VAL_PATH)
TEST_PATH_LOCAL  = to_local_dbfs(TEST_PATH)
print("Train:", TRAIN_PATH_LOCAL, "\nVal:", VAL_PATH_LOCAL, "\nTest:", TEST_PATH_LOCAL)

# Use VAL if it exists; otherwise fall back to TEST during epochs
USE_VAL = os.path.exists(VAL_PATH_LOCAL)
print("Per-epoch evaluation on:", ("VAL" if USE_VAL else "TEST"))

with open(META_PATH,"r") as f:
    META = json.load(f)
FEATURES_FOR_LSTM = META["FEATURES_FOR_LSTM"]
N_NUM   = int(META["N_NUM"])
CAT_IDX_COLS = META["CAT_IDX_COLS"]
cat_cards     = META["cat_cardinalities"]
INPUT_SIZE    = len(FEATURES_FOR_LSTM)

def emb_dim(n: int) -> int:
    v = int(math.sqrt(int(n))) * 2
    return builtins.max(2, builtins.min(64, v))

EMB_SPECS = [(name, int(cat_cards[name]), emb_dim(int(cat_cards[name]))) for name in CAT_IDX_COLS]
print("EMB_SPECS:", EMB_SPECS)

BATCH_SIZE, SCAN_BATCH = 128, 2048
HIDDEN, LAYERS, DROPOUT = 128, 1, 0.1
LR, EPOCHS, PATIENCE = 2e-3, 8, 2

def minibatches_from_parquet(folder_local, batch_size=BATCH_SIZE, scan_batch=SCAN_BATCH, shuffle_files=True):
    dset = ds.dataset(folder_local, format="parquet")
    frags = list(dset.get_fragments())
    if shuffle_files: np.random.shuffle(frags)
    for frag in frags:
        scanner = ds.Scanner.from_fragment(frag, columns=["features","label"], batch_size=scan_batch)
        for rec in scanner.to_reader():
            X = np.asarray(rec["features"].to_pylist(), dtype=np.float32)
            y = np.asarray(rec["label"].to_pylist(),    dtype=np.float32)
            n = X.shape[0]; idx = np.arange(n); np.random.shuffle(idx)
            for i in range(0, n, batch_size):
                j = idx[i:i+batch_size]
                yield X[j], y[j]

@torch.no_grad()
def evaluate_with_bar(path_local, model):
    P, Y = [], []
    dset = ds.dataset(path_local, format="parquet")
    frag_list = list(dset.get_fragments())
    bar = tqdm(total=len(frag_list), desc="Eval", leave=False, ncols=0)
    for frag in frag_list:
        scanner = ds.Scanner.from_fragment(frag, columns=["features","label"], batch_size=SCAN_BATCH, use_threads=False)
        for rec in scanner.to_reader():
            X = np.asarray(rec["features"].to_pylist(), dtype=np.float32)
            y = np.asarray(rec["label"].to_pylist(),    dtype=np.float32)
            for i in range(0, X.shape[0], BATCH_SIZE):
                Xt = torch.from_numpy(X[i:i+BATCH_SIZE]).to(DEVICE, non_blocking=True)
                P.append(torch.sigmoid(model(Xt)).cpu().numpy().astype(np.float32))
                Y.append(y[i:i+BATCH_SIZE])
    bar.close()
    if not P:
        return dict(auc=float("nan"), ap=float("nan"), acc=float("nan"), f1=float("nan"))
    p = np.concatenate(P); y = np.concatenate(Y)
    yhat = (p >= 0.5).astype(int)
    def safe(fn, *a, **k):
        try: return fn(*a, **k)
        except: return float("nan")
    return dict(
        auc=safe(roc_auc_score, y, p),
        ap=safe(average_precision_score, y, p),
        acc=float((yhat==y).mean()),
        f1=safe(f1_score, y, yhat),
    )

class LSTMWithEmb(nn.Module):
    def __init__(self, n_num, emb_specs, hidden=128, layers=1, dropout=0.1, bidirectional=False):
        super().__init__()
        self.emb_names = [n for n,_,_ in emb_specs]
        self.embs = nn.ModuleDict({n: nn.Embedding(card, dim) for n,card,dim in emb_specs})
        emb_total = builtins.sum([dim for _,_,dim in emb_specs])
        in_size = n_num + emb_total
        self.lstm = nn.LSTM(in_size, hidden, num_layers=layers,
                            dropout=(dropout if layers>1 else 0.0),
                            batch_first=True, bidirectional=bidirectional)
        out = hidden * (2 if bidirectional else 1)
        self.head = nn.Sequential(nn.LayerNorm(out), nn.Linear(out, 1))
    def forward(self, x):
        x_num = x[:, :, :N_NUM]
        x_cat = x[:, :, N_NUM:]
        cat_embs = [ self.embs[name](x_cat[:,:,i].long().clamp(min=0)) for i,name in enumerate(self.emb_names) ]
        xin = torch.cat([x_num, torch.cat(cat_embs, dim=-1)], dim=-1) if cat_embs else x_num
        _, (hn, _) = self.lstm(xin)
        return self.head(hn[-1]).squeeze(1)

model = LSTMWithEmb(N_NUM, EMB_SPECS, hidden=HIDDEN, layers=LAYERS, dropout=DROPOUT).to(DEVICE)
opt = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
loss_fn = nn.BCEWithLogitsLoss()

# (optional) estimated batches
def count_rows_local(parquet_path_local: str) -> int:
    dset = ds.dataset(parquet_path_local, format="parquet")
    return builtins.sum(frag.count_rows() for frag in dset.get_fragments())
try:
    n_train = count_rows_local(TRAIN_PATH_LOCAL)
    est_batches = int(math.ceil(n_train / BATCH_SIZE)) if n_train else None
except Exception:
    est_batches = None

BEST_AUC, BAD = -1.0, 0
for ep in range(1, EPOCHS+1):
    model.train()
    bar = tqdm(total=est_batches or 0, desc=f"Epoch {ep}/{EPOCHS}", leave=False, ncols=0, file=sys.stdout)
    b = 0
    for X, y in minibatches_from_parquet(TRAIN_PATH_LOCAL, batch_size=BATCH_SIZE, scan_batch=SCAN_BATCH, shuffle_files=True):
        Xt = torch.from_numpy(X).to(DEVICE, non_blocking=True)
        yt = torch.from_numpy(y).to(DEVICE, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        loss = loss_fn(model(Xt), yt)
        loss.backward(); opt.step()
        b += 1
        if est_batches: bar.update(1)
        if (b % 50) == 0:
            del Xt, yt; gc.collect()
            if DEVICE == "cuda": torch.cuda.empty_cache()
    bar.close()

    # --- minimal change: prefer VAL, else TEST ---
    eval_path = VAL_PATH_LOCAL if USE_VAL else TEST_PATH_LOCAL
    m = evaluate_with_bar(eval_path, model)
    print(f"Epoch {ep:02d} — AUC {m['auc']:.4f} | AP {m['ap']:.4f} | ACC {m['acc']:.4f} | F1 {m['f1']:.4f}", flush=True)

    if not math.isnan(m["auc"]) and m["auc"] > BEST_AUC + 1e-4:
        BEST_AUC, BAD = m["auc"], 0
        torch.save(model.state_dict(), "/dbfs/models_lstm_indexers_v1/lstm_best.pt")
        print("✔ saved new best", flush=True)
    else:
        BAD += 1
        print(f"No improvement ({BAD}/{PATIENCE})", flush=True)
        if BAD >= PATIENCE:
            print(f"Early stop. Best AUC={BEST_AUC:.4f}", flush=True)
            break

print("Training finished. Best checkpoint: /dbfs/models_lstm_indexers_v1/lstm_best.pt", flush=True)


In [0]:
# ==== Recreate model, load best checkpoint, pick threshold on VAL, evaluate on TEST ====
import os, json, math, numpy as np
import builtins, torch, torch.nn as nn
import pyarrow.dataset as ds
from sklearn.metrics import precision_recall_curve, confusion_matrix, roc_auc_score, average_precision_score, f1_score, accuracy_score

# ---- NEW: safe builtins alias for round/min/max to avoid Spark shadowing
import builtins as _py

# --- paths/helpers (match your training run) ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
def to_local_dbfs(p: str) -> str: return p.replace("dbfs:/","/dbfs/")
META_PATH = "/dbfs/models/lstm_indexers_v1/meta.json"
TRAIN_PATH_LOCAL = to_local_dbfs(TRAIN_PATH)
VAL_PATH_LOCAL   = to_local_dbfs(VAL_PATH)
TEST_PATH_LOCAL  = to_local_dbfs(TEST_PATH)
USE_VAL = os.path.exists(VAL_PATH_LOCAL)

# --- load meta to rebuild shapes/embeddings ---
with open(META_PATH, "r") as f:
    META = json.load(f)
FEATURES_FOR_LSTM = META["FEATURES_FOR_LSTM"]
N_NUM             = int(META["N_NUM"])
CAT_IDX_COLS      = META["CAT_IDX_COLS"]            # e.g. ["ID_INDEX","CONCELHO_INDEX"]
cat_cards         = META["cat_cardinalities"]       # dict with _INDEX keys
INPUT_SIZE        = len(FEATURES_FOR_LSTM)

def emb_dim(n: int) -> int:
    v = int(math.sqrt(int(n))) * 2
    return _py.max(2, _py.min(64, v))   # <-- use Python builtins

# ensure keys align
cat_cards_idx = { (k if k.endswith("_INDEX") else f"{k}_INDEX"): int(v)
                  for k,v in cat_cards.items() }
EMB_SPECS = [(name, int(cat_cards_idx[name]), emb_dim(int(cat_cards_idx[name])))
             for name in CAT_IDX_COLS]

# --- model definition must match training ---
class LSTMWithEmb(nn.Module):
    def __init__(self, n_num, emb_specs, hidden=128, layers=1, dropout=0.1, bidirectional=False):
        super().__init__()
        self.emb_names = [n for n,_,_ in emb_specs]
        self.embs = nn.ModuleDict({n: nn.Embedding(card, dim) for n,card,dim in emb_specs})
        emb_total = _py.sum([dim for _,_,dim in emb_specs])  # <-- builtins.sum
        in_size = n_num + emb_total
        self.lstm = nn.LSTM(in_size, hidden, num_layers=layers,
                            dropout=(dropout if layers>1 else 0.0),
                            batch_first=True, bidirectional=bidirectional)
        out = hidden * (2 if bidirectional else 1)
        self.head = nn.Sequential(nn.LayerNorm(out), nn.Linear(out, 1))
    def forward(self, x):
        x_num = x[:, :, :N_NUM]
        x_cat = x[:, :, N_NUM:]
        cat_embs = [ self.embs[name](x_cat[:,:,i].long().clamp(min=0)) for i,name in enumerate(self.emb_names) ]
        xin = torch.cat([x_num, torch.cat(cat_embs, dim=-1)], dim=-1) if cat_embs else x_num
        _, (hn, _) = self.lstm(xin)
        return self.head(hn[-1]).squeeze(1)

# --- instantiate with same HPs used in training, then load weights ---
HIDDEN, LAYERS, DROPOUT = 128, 1, 0.1
model = LSTMWithEmb(N_NUM, EMB_SPECS, hidden=HIDDEN, layers=LAYERS, dropout=DROPOUT).to(DEVICE)
state = torch.load("/dbfs/models_lstm_indexers_v1/lstm_best.pt", map_location=DEVICE)
model.load_state_dict(state)
model.eval()

# --- inference helpers ---
@torch.no_grad()
def collect_probs(folder_local, model, batch_size=128, scan_batch=2048, device=DEVICE):
    P, Y = [], []
    dset = ds.dataset(folder_local, format="parquet")
    for frag in dset.get_fragments():
        scanner = ds.Scanner.from_fragment(frag, columns=["features","label"], batch_size=scan_batch, use_threads=False)
        for rec in scanner.to_reader():
            X = np.asarray(rec["features"].to_pylist(), dtype=np.float32)
            y = np.asarray(rec["label"].to_pylist(),    dtype=np.float32)
            for i in range(0, X.shape[0], batch_size):
                Xt = torch.from_numpy(X[i:i+batch_size]).to(device, non_blocking=True)
                P.append(torch.sigmoid(model(Xt)).cpu().numpy().astype(np.float32))
                Y.append(y[i:i+batch_size])
    if not P: return np.array([]), np.array([])
    return np.concatenate(P), np.concatenate(Y)

def eval_at_threshold(p, y, thr):
    yhat = (p >= thr).astype(int)
    # ---- NEW: make CM robust when a class is missing
    tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0,1]).ravel()
    return {
        "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn),
        "acc": float((yhat==y).mean()),
        "f1":  f1_score(y, yhat) if (tp+fp>0 and tp+fn>0) else 0.0,
        "prec": float(tp/(tp+fp)) if (tp+fp)>0 else 0.0,
        "rec":  float(tp/(tp+fn)) if (tp+fn)>0 else 0.0,
        "thr": float(thr),
        "auc": roc_auc_score(y, p) if len(np.unique(y))>1 else float("nan"),
        "ap":  average_precision_score(y, p) if len(np.unique(y))>1 else float("nan"),
    }

# pretty-printer that avoids Spark's round() shadowing
def pretty(d):
    out = {}
    for k, v in d.items():
        if isinstance(v, (float, np.floating)):
            out[k] = float(v) if not np.isfinite(v) else _py.round(float(v), 4)
        else:
            out[k] = v
    return out

# --- pick threshold on VAL (fallback to TEST if no VAL) ---
eval_path_local = VAL_PATH_LOCAL if USE_VAL else TEST_PATH_LOCAL
p_val, y_val = collect_probs(eval_path_local, model)

pr, rc, thr = precision_recall_curve(y_val, p_val)    # thr length = len(pr)-1
f1 = (2*pr*rc)/(pr+rc+1e-12)
j  = int(np.nanargmax(f1))
# Map from PR index to threshold index (thr has len(pr)-1)
thr_idx = (_py.max(_py.min(j-1, len(thr)-1), 0) if len(thr) > 0 else None)
thr_best_f1 = float(thr[thr_idx]) if thr_idx is not None else 0.5

val_best = eval_at_threshold(p_val, y_val, thr_best_f1)
print("VAL — best F1:", pretty(val_best))

# Optional: target recall threshold
TARGET_REC = 0.90
idxs = np.where(rc >= TARGET_REC)[0]
if len(idxs) > 0 and len(thr) > 0:
    thr_idx_rec = _py.max(int(idxs[0]) - 1, 0)
    thr_rec = float(thr[thr_idx_rec])
else:
    thr_rec = thr_best_f1

val_rec = eval_at_threshold(p_val, y_val, thr_rec)
print("VAL — target recall:", TARGET_REC, pretty(val_rec))

# --- evaluate on TEST at the chosen threshold (choose one) ---
CHOSEN_THR = thr_best_f1  # or thr_rec
p_test, y_test = collect_probs(TEST_PATH_LOCAL, model)
test_metrics = eval_at_threshold(p_test, y_test, CHOSEN_THR)
print("TEST @ chosen thr:", pretty(test_metrics))


In [0]:
# compute pos_weight on TRAIN parquet once
import pyarrow.dataset as ds, numpy as np

def label_counts(path_local):
    d = ds.dataset(path_local, format="parquet")
    pos = 0; tot = 0
    for frag in d.get_fragments():
        for rec in ds.Scanner.from_fragment(frag, columns=["label"]).to_reader():
            y = np.asarray(rec["label"].to_pylist(), dtype=np.float32)
            pos += (y == 1).sum(); tot += y.size
    return int(pos), int(tot)

pos, tot = label_counts(TRAIN_PATH_LOCAL)
neg = tot - pos
pos_weight = (neg / max(pos, 1.0))
print("TRAIN pos_weight ≈", pos_weight)

# in your training cell, replace BCE loss with:
loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight, device=DEVICE, dtype=torch.float32))
